In [ ]:
import sqlite3

DB_PATH = r"C:\Users\Mackay Iria\DataMiningProjects\telecom-edge-dw\database\telecom_warehouse.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print("🛠️ Starting Data Modeling Relational Split...\n")

# 1. Enable Foreign Key Constraints in SQLite (turned off by default)
cursor.execute("PRAGMA foreign_keys = ON;")

# 2. Create the Customer Dimension Table (Primary Key enforced)
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_customers (
    customer_id TEXT PRIMARY KEY,
    first_name TEXT,
    last_name TEXT,
    customer_email TEXT
);
""")

# 3. Create the Plan Dimension Table (Primary Key enforced)
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_plans (
    plan_id INTEGER PRIMARY KEY AUTOINCREMENT,
    plan_type TEXT UNIQUE
);
""")

# 4. Create the Relational Fact Invoice Table (Foreign Keys pointing to dimensions)
cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_invoices (
    invoice_id TEXT PRIMARY KEY,
    invoice_date TEXT,
    customer_id TEXT,
    plan_id INTEGER,
    region TEXT,
    total_invoice_amount REAL,
    payment_status TEXT,
    payment_method TEXT,
    FOREIGN KEY (customer_id) REFERENCES dim_customers(customer_id),
    FOREIGN KEY (plan_id) REFERENCES dim_plans(plan_id)
);
""")
print("✓ Relational table skeletons provisioned.")

# =========================================================================
# POPULATING THE DIMENSIONS AND FACTS (ETL Data Distribution)
# =========================================================================

# Populate dim_customers (Using INSERT OR IGNORE to prevent duplicate errors)
cursor.execute("""
INSERT OR IGNORE INTO dim_customers (customer_id, first_name, last_name, customer_email)
SELECT DISTINCT customer_id, first_name, last_name, customer_email 
FROM analytics_cleaned_invoices;
""")

# Populate dim_plans
cursor.execute("""
INSERT OR IGNORE INTO dim_plans (plan_type)
SELECT DISTINCT plan_type 
FROM analytics_cleaned_invoices;
""")

# Populate fact_invoices by JOINING your flat table to our new dimensions map
cursor.execute("""
INSERT OR IGNORE INTO fact_invoices (
    invoice_id, invoice_date, customer_id, plan_id, region, 
    total_invoice_amount, payment_status, payment_method
)
SELECT 
    a.invoice_id,
    a.invoice_date,
    a.customer_id,
    p.plan_id,
    a.region,
    a.total_invoice_amount,
    a.payment_status,
    a.payment_method
FROM analytics_cleaned_invoices a
JOIN dim_plans p ON a.plan_type = p.plan_type;
""")

conn.commit()
print("✓ Data successfully distributed into Star Schema modeling rules.")

# List out the updated table count to verify
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("\n🗄️ Current tables inside your SQLite Warehouse:", [r[0] for r in cursor.fetchall()])

conn.close()
print("\n🏁 Modeling sequence complete!")
